In [136]:
import importlib
import rww_pytorch_model
importlib.reload(rww_pytorch_model)

from rww_pytorch_model import ParamsJR
from rww_pytorch_model import RNNWWD
from rww_pytorch_model import Model_fitting
import matplotlib.pyplot as plt # for plotting
import numpy as np # for numerical operations
import pandas as pd # for data manipulation
import seaborn as sns # for plotting 
import time # for timer
from datetime import datetime
from nilearn import image
import pickle
import itertools
import os

In [102]:
#generate results csv
df_res = pd.DataFrame(columns = ['sub_id', 'session', 'stim_site', 'stim_type', 'parcellation', 'max_parcels', 'num_epochs', 'gsr', 'r'])
df_res.to_csv('../res/sim_results.csv')

In [169]:
def file_paths(stim_site, sub_id, parcellation, session, gsr = None):

    sub_id = str(sub_id)
    session = str(session)

    if 'B' in sub_id or 'S' in sub_id:
        stim_type = 'Sham'
    else:
        stim_type = 'Active'


    ses_path = '../data/' + stim_site + '/' + stim_type + '/' + sub_id + '/ses-0' + session
    atlas_path = ses_path + '/atlases/' + sub_id + '_' + parcellation + '.nii.gz'

    if stim_site == 'DLPFC' or stim_site == 'RIFG':
        bold_path = ses_path + '/REST/sub-' + sub_id[1:] + '_ses-0' + session + '_' + ((not gsr) * "no") + 'GSR_Filtered_4DVolume.nii'
        sc_path =  ses_path + '/dMRI/cm_nodes_' + parcellation + '.nii.gz_count.csv'   
    elif stim_site == 'PSTS':
        bold_path = ses_path + '/REST/' + sub_id[1:] + '_0' + session + '_R_MEICA_T1c_medn_nlw.nii.gz'
        sc_path =  ses_path + '/dMRI/cm_nodes_' + parcellation + '_count.csv'

    return sc_path, bold_path, atlas_path


def model_fit(stim_site, session, sub_id, parcellation, max_parcels, num_epochs = 20, save = False, csv_path = None, gsr = None):

    start_time = time.time()

    
    sc_path, bold_path, atlas_path = file_paths(stim_site = stim_site, sub_id = sub_id, parcellation = parcellation, 
                                                session = session, gsr = gsr)

    #structural
    #sc_file = '../dlPFC/sub-4602/anat/DWI/cm_nodes_Schaefer100+Tian+Suit.nii.gz_count.csv'
    sc = np.loadtxt(sc_path, delimiter = ',')[0:max_parcels, 0:max_parcels]#[0:100, 0:100]

    #print(np.shape(sc), np.max(sc), np.min(sc))

    sc = sc -np.diag(np.diag(sc))
    sc = 0.5*(sc.T+sc)
    sc = np.log1p(sc)/np.linalg.norm(np.log1p(sc))
    print(np.shape(sc))

    dist = np.nan_to_num(1/sc)
    dist[dist> 200] = 0

    #functional
    #bold = image.load_img('../dlPFC/sub-4602/func/sub-4602_ses-01_task-rest_space-MNI152NLin6Asym_res-02_desc-denoisedSmoothed_bold.nii.gz')
    bold = image.load_img(bold_path)
    bold_data = image.get_data(bold)

    atlas = image.load_img(atlas_path)
    atlas = image.resample_to_img(atlas, image.index_img(bold, 1), interpolation = 'nearest')
    atlas_data = image.get_data(atlas)
    atlas_data[atlas_data > max_parcels] = 0 #only looking at schaefer 100
    n_parcels = int(np.max(atlas_data))
    n_time = int(np.shape(bold_data)[3])

    ts = np.zeros(((n_time, n_parcels)))
    for i_parcel in range(n_parcels):
        for i_time in range(n_time):
            ts[i_time, i_parcel] = np.mean(bold_data[:,:,:,i_time][atlas_data == i_parcel + 1])
    ts = ts/np.max(ts)
    #ts = (ts - np.mean(ts))/np.std(ts) #trying z scores to account for differences in baseline scanner intensity
    
    
    fc_emp = np.corrcoef(ts.T)

    #print(fc_emp)


    node_size = n_parcels
    output_size = n_parcels
    #num_epochs = 200
    #batch_size =20
    batch_size = 16 #trying 16, divides cleanly into 176 time points
    step_size = .05
    input_size = 2
    tr = 2.5 #changed

    ## call WWD module
    par = ParamsJR("WWD", g = [80,.2], g_EE= [0.5, 5], g_IE= [0.5, 5], \
                        g_EI= [0.16963709468378746, 10], I_0=[0.39410256410256406, 10],\
                        std_in=[0.02, 0],  mu = [2.5, 5])



    """par = ParamsJR("WWD", g = [20, 0], g_EE= [3.393939393939394, 0], g_IE= [10.260801579764808, 0], \
                        g_EI= [0.3290554714188569, 0], I_0= [0.39410256410256406, 0], tau_E =[0.05, 0], tau_I =[ 0.01, 0], gamma_E= [0.641, 0], gamma_I = [1, 0],\
                        std_in=[0.05, 0],  mu = [5.5, 0])# Get the WWD model module for forward in a batch.""" 
    """par = ParamsJR("WWD", g = [2, 0], g_EE= [3.5, 0], g_IE= [2.5, 0], \
                        g_EI= [3.75, 0], I_0= [0.382, 0], tau_E =[ 0.01, 0], tau_I =[ 0.02, 0], gamma_E= [0.641, 0], gamma_I = [1, 0],\
                        std_in=[0.01, 0],  mu = [5.5, 0])# Get the WWD model module for forward in a batch."""
    model = RNNWWD(input_size, node_size, batch_size, step_size, tr, sc, dist, True,  par)





    data_mean = [ts.T]*num_epochs
    data_mean =np.array(data_mean)
    # call model fit method
    #lr is 0.001 by default
    F = Model_fitting(model, data_mean, num_epochs, 1, 0.001)

    # fit data(train)

    output_train = F.train()

    X0 = 0.45 * np.random.uniform(0, 1, (node_size, 6)) + np.array(
                [0, 0, 0, 1.0, 1.0, 1.0])
    hE0 = np.random.uniform(0, 5, (node_size, 500))
    base_batch_num = 20
    output_test = F.test(X0, hE0, base_batch_num)
    #output_test = F.test(120)
    fc_test = np.corrcoef(F.output_sim.bold_test)

    fig, ax = plt.subplots(1,3, figsize=(12,8))
    ax[1].imshow(fc_emp-np.diag(np.diag(fc_emp)), cmap='bwr')
    ax[1].set_title('Empirical FC')
    ax[0].plot(F.output_sim.bold_test.T)
    ax[0].set_title('Test')
    ax[2].imshow(fc_test-np.diag(np.diag(fc_test)), cmap='bwr')
    ax[2].set_title('Test FC')
    

    if save:

        if 'B' in sub_id or 'S' in sub_id:
            stim_type = 'Sham'
        else:
            stim_type = 'Active'
        
        df = pd.read_csv(csv_path, header = 0, index_col = 0)
        desc = sub_id + '_ses-0' + str(session) + '_' + parcellation + '_parcels' +  str(max_parcels) + '_epoch' + str(num_epochs)
        if stim_site == 'DLPFC' or stim_site == 'RIFG':
            desc += '_' + ((not gsr) * 'no') + 'GSR'

        sub_dir = '../res/' + stim_site + '/' + sub_id
        if not os.path.isdir(sub_dir):
            os.makedirs(sub_dir)

        plt.savefig(sub_dir + '/' + desc + '.png')
        np.savetxt(sub_dir + '/' + desc + '_fc_emp.csv', fc_emp)
        np.savetxt(sub_dir + '/' + desc + '_fc_test.csv', fc_test)
        r = np.corrcoef(fc_emp.flatten(), fc_test.flatten())[0,1]
        df.loc[desc] = [sub_id, session, stim_site, stim_type, parcellation, max_parcels, num_epochs, gsr, r]
        df.to_csv(csv_path)

    plt.show()

    end_time =  time.time()
    print('running time is  {0} \'s'.format(end_time - start_time ))

def iterate(stim_sites, sessions, parcellations, epochs, csv_path, use_gsr = [True, False], check_csv = True):
    current_time = datetime.now()
    df_error = pd.DataFrame(columns = ['message'])
    
    for stim_site, session, parcellation, num_epochs in itertools.product(stim_sites, sessions, parcellations, epochs):
        for sub_id, max_parcels, gsr in itertools.product(stim_sites[stim_site], parcellations[parcellation], use_gsr):
            
            df = pd.read_csv(csv_path, header = 0, index_col = 0)
            desc = sub_id + '_ses-0' + str(session) + '_' + parcellation + '_parcels' +  str(max_parcels) + '_epoch' + str(num_epochs)
            if stim_site == 'DLPFC' or stim_site == 'RIFG':
                desc += '_' + ((not gsr) * 'no') + 'GSR'
            elif stim_site == 'PSTS':
                gsr = None

            
            run_participant = True
            if check_csv:
                run_participant = not desc in df.index

            if run_participant:
                print(sub_id,session,stim_site,parcellation,max_parcels,num_epochs,gsr)
                try:
                    model_fit(stim_site = stim_site, session = session, sub_id = sub_id, parcellation = parcellation,
                          max_parcels = max_parcels, num_epochs = num_epochs, save = True,  csv_path = csv_path, gsr = gsr)
                except Exception as e:
                    df_error.loc[desc, 'message'] = str(e)

    if np.shape(df_error)[0] > 0: #more than 0 rows
        timestamp = str(current_time.year) + str(current_time.month) + str(current_time.day) + '_' + str(current_time.hour) + str(current_time.minute)
        df_error.to_csv('../temp/errors/' + timestamp + '_errors.csv')


In [ ]:
model_fit(stim_site = 'PSTS', session = 2, sub_id = 'B941321', parcellation = 'Schaefer100+Tian+Suit', max_parcels = 100, num_epochs = 1, save = True, csv_path = '../res/sim_results.csv', gsr = True)


FileNotFoundError: ../data/PSTS/Sham/B9413211/ses-02/dMRI/cm_nodes_Schaefer100+Tian+Suit_count.csv not found.

In [112]:
os.makedirs('../res/DLPFC/A4602')

In [168]:
iterate(stim_sites = {'PSTS':['A941320', 'A941366', 'B941321', 'B941347'], 'DLPFC':['A4602', 'A4608', 'S4603', 'S4611']}, sessions = [1,2], parcellations = {'Schaefer100+Tian+Suit':[100, 184], 'Schaefer200+Tian+Suit':[284]}, epochs = [20], csv_path = '../res/sim_results.csv', use_gsr = [True, False], check_csv = True)

A941320 1 PSTS Schaefer100+Tian+Suit 100 20 None
A941320 1 PSTS Schaefer100+Tian+Suit 100 20 None
A941320 1 PSTS Schaefer100+Tian+Suit 184 20 None
A941320 1 PSTS Schaefer100+Tian+Suit 184 20 None
A941366 1 PSTS Schaefer100+Tian+Suit 100 20 None
A941366 1 PSTS Schaefer100+Tian+Suit 100 20 None
A941366 1 PSTS Schaefer100+Tian+Suit 184 20 None
A941366 1 PSTS Schaefer100+Tian+Suit 184 20 None
B941321 1 PSTS Schaefer100+Tian+Suit 100 20 None
B941321 1 PSTS Schaefer100+Tian+Suit 100 20 None
B941321 1 PSTS Schaefer100+Tian+Suit 184 20 None
B941321 1 PSTS Schaefer100+Tian+Suit 184 20 None
B941347 1 PSTS Schaefer100+Tian+Suit 100 20 None
B941347 1 PSTS Schaefer100+Tian+Suit 100 20 None
B941347 1 PSTS Schaefer100+Tian+Suit 184 20 None
B941347 1 PSTS Schaefer100+Tian+Suit 184 20 None
A941320 1 PSTS Schaefer200+Tian+Suit 284 20 None
A941320 1 PSTS Schaefer200+Tian+Suit 284 20 None
A941366 1 PSTS Schaefer200+Tian+Suit 284 20 None
A941366 1 PSTS Schaefer200+Tian+Suit 284 20 None
B941321 1 PSTS Schae

In [95]:
test = True
type('_' + ((not test) * 'no') + 'GSR')
#'_' + (not test * 'no') + 'GSR'

str

In [94]:
(False * 'no')

''